# BDCBot — Bangladesh Constitution Chatbot (development notebook)

Pipeline: **PDF → structure-aware parsing (Parts, Chapters, Articles, sidebar headings) → article chunks →
Sentence-BERT embeddings → FAISS → hybrid legal retrieval → Gemini answer (grounded) → out-of-scope handling → memory.**

This notebook is where the pipeline was built and tested. The finished software (backend API, chat frontend, auth,
logging, knowledge-base management) lives in the `backend/` and `frontend/` folders of the repository.

Run the cells top to bottom (Runtime → Run all).

# 1. Setup

In [ ]:
!pip install -q sentence-transformers faiss-cpu pymupdf google-genai

In [ ]:
import fitz  # PyMuPDF
import re
import json
from collections import Counter

# 2. Load Knowledge Base

In [ ]:
import os

# Upload the PDF in the Colab file panel, or run this cell and pick the file.
path = "/content/constitution-bangladesh.pdf"
if not os.path.exists(path) and os.path.exists(path + ".pdf"):
    path = path + ".pdf"          # Colab sometimes saves it as .pdf.pdf

if not os.path.exists(path):
    try:
        from google.colab import files
        uploaded = files.upload()
        path = "/content/" + next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put the constitution PDF at {path}")

print("Using:", path)

# 3. Text Extraction

In [ ]:
pdf_path = path

doc = fitz.open(pdf_path)

text = ""

for page in doc:
    text += page.get_text()

print("Number of characters:", len(text))
print(text[:3000])

**We might need text cleaning here**

In [ ]:
print("Total pages:", len(doc))

for i in range(min(10, len(doc))):
    page_text = doc[i].get_text()
    print(f"\n{'='*80}")
    print(f"PAGE {i+1}")
    print(page_text[:1000])

**01.10.26**

In [ ]:
font_stats = Counter()

for page_num in range(min(70, len(doc))):
    page = doc[page_num]

    for block in page.get_text("dict")["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            for span in line["spans"]:
                span_text = span["text"].strip()

                if span_text:
                    font_stats[
                        (
                            round(span["size"], 1),
                            span["font"],
                            span["flags"]
                        )
                    ] += 1

for item, count in font_stats.most_common(30):
    print(item, "→", count)

In [ ]:
def get_spans(doc, start_page=0, end_page=None):

    spans = []
    end_page = min(end_page or 70, len(doc))

    for page_num in range(start_page, end_page):

        page = doc[page_num]
        page_dict = page.get_text("dict")

        for block in page_dict["blocks"]:

            if "lines" not in block:
                continue

            for line in block["lines"]:

                for span in line["spans"]:

                    txt = span["text"].strip()

                    if not txt:
                        continue

                    spans.append({
                        "page": page_num,
                        "text": txt,
                        "x0": span["bbox"][0],
                        "y0": span["bbox"][1],
                        "x1": span["bbox"][2],
                        "y1": span["bbox"][3],
                        "size": span["size"],
                        "font": span["font"],
                        "flags": span["flags"]
                    })

    return spans


spans = get_spans(doc)

print("Total spans:", len(spans))

In [ ]:
def is_bold(span):

    font = span["font"].lower()
    flags = span["flags"]

    return (
        "bold" in font
        or "black" in font
        or "semibold" in font
        or bool(flags & 16)
    )

In [ ]:
for s in spans[:100]:
    print(
        f"PAGE={s['page']+1:02d} "
        f"SIZE={s['size']:.1f} "
        f"BOLD={is_bold(s)} "
        f"TEXT={s['text']}"
    )

In [ ]:
for s in spans:
  if s['page'] == 16:
    print(
        f"PAGE={s['page']+1:02d} "
        f"SIZE={s['size']:.1f} "
        f"BOLD={is_bold(s)} "
        f"TEXT={s['text']}"
    )

**01.10.26 Run 2**

In [ ]:
START_PAGE = 0
END_PAGE = min(70, len(doc))  # articles end on page 70; pages 71-75 are the amendment notes (section 3b)

# ---------- Helpers ----------

def is_bold(span):
    font = span["font"].lower()
    flags = span["flags"]
    return (
        "bold" in font
        or "black" in font
        or "semibold" in font
        or bool(flags & 16)
    )

def near(value, target, tolerance=0.5):
    return abs(value - target) <= tolerance

def clean_text(text):
    text = text.replace("\u200c", "").replace("\u200d", "")
    text = re.sub(r"02/02/2022", "", text)
    text = re.sub(
        r"The Constitution of the People.?s Republic of Bangladesh",
        "",
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

# ---------- Continuous text ----------
# This is used ONLY for authoritative article boundaries.

page_texts = []

for page_num in range(START_PAGE, END_PAGE):
    page_text = clean_text(doc[page_num].get_text())
    page_texts.append(page_text)

text = "\n".join(page_texts)

article_pattern = re.compile(
    r'(?m)^\s*\[?(\d{1,3}[A-Z]?)\.\s+(?=[A-Z\[\(])'
)

matches = list(article_pattern.finditer(text))
article_numbers = [m.group(1) for m in matches]

print("Detected articles:", len(article_numbers))
print(article_numbers)

# ---------- PDF lines with formatting + coordinates ----------

lines = []

for page_num in range(START_PAGE, END_PAGE):

    page = doc[page_num]

    for block in page.get_text("dict")["blocks"]:

        if "lines" not in block:
            continue

        for line in block["lines"]:

            spans = line["spans"]

            if not spans:
                continue

            parts = [
                s["text"].strip()
                for s in spans
                if s["text"].strip()
            ]

            if not parts:
                continue

            line_text = clean_text(" ".join(parts))

            if not line_text:
                continue

            lines.append({
                "page": page_num + 1,
                "y0": line["bbox"][1],
                "y1": line["bbox"][3],
                "x0": line["bbox"][0],
                "x1": line["bbox"][2],
                "text": line_text,
                "size": sum(s["size"] for s in spans) / len(spans),
                "bold": any(is_bold(s) for s in spans)
            })

# Remove header and standalone page numbers

content_lines = []

for line in lines:

    if near(line["size"], 5.3):
        continue

    if (
        near(line["size"], 7.5)
        and re.fullmatch(r"\d+", line["text"])
    ):
        continue

    content_lines.append(line)

print("Usable lines:", len(content_lines))

In [ ]:
# ============================================================
# CELL 2 — Detect Parts, Chapters, Sidebar Headings & Articles
# ============================================================

MAIN_X_MIN = 100
SIDEBAR_X_MAX = 100


# ------------------------------------------------------------
# PART / CHAPTER DETECTION
# ------------------------------------------------------------

def is_part_number(line):
    return (
        near(line["size"], 13.5, 0.6)
        and line["bold"]
        and bool(re.fullmatch(
            r"PART\s+[IVXLCDM]+A?",
            line["text"],
            re.I
        ))
    )


def is_chapter_number(line):
    return (
        near(line["size"], 13.5, 0.6)
        and line["bold"]
        and bool(re.fullmatch(
            r"CHAPTER\s+[IVXLCDM]+",
            line["text"],
            re.I
        ))
    )


def is_part_title(line):
    return (
        line["x0"] >= MAIN_X_MIN
        and near(line["size"], 12.0, 0.6)
        and not line["bold"]
    )


def is_chapter_title(line):
    return (
        line["x0"] >= MAIN_X_MIN
        and near(line["size"], 12.0, 0.6)
        and line["bold"]
    )


# ------------------------------------------------------------
# SIDEBAR / ARTICLE DETECTION
# ------------------------------------------------------------

def is_sidebar_heading(line):
    return (
        near(line["size"], 12.0, 0.6)
        and line["bold"]
        and line["x0"] < SIDEBAR_X_MAX
    )


def is_article_start(line):
    return (
        line["x0"] >= MAIN_X_MIN
        and bool(re.match(
            r'^\[?(\d{1,3}[A-Z]?)\.',
            line["text"]
        ))
    )


# ------------------------------------------------------------
# FIND THE TITLE FOLLOWING A STRUCTURAL NUMBER
# ------------------------------------------------------------

def find_structural_title(index, title_type):
    current = content_lines[index]

    # First search on the same page
    for j in range(index + 1, min(index + 6, len(content_lines))):
        candidate = content_lines[j]

        if candidate["page"] != current["page"]:
            break

        if is_part_number(candidate) or is_chapter_number(candidate):
            break

        if title_type == "part" and is_part_title(candidate):
            return candidate["text"]

        if title_type == "chapter" and is_chapter_title(candidate):
            return candidate["text"]


    # --------------------------------------------------------
    # If the structural number is near the bottom of a page,
    # the title may appear at the top of the next page.
    # --------------------------------------------------------

    next_page = current["page"] + 1

    next_page_lines = [
        x for x in content_lines
        if x["page"] == next_page
        and x["x0"] >= MAIN_X_MIN
    ]

    # Only inspect the upper portion of the next page.
    next_page_lines = [
        x for x in next_page_lines
        if x["y0"] < 150
    ]

    next_page_lines.sort(key=lambda x: x["y0"])

    for candidate in next_page_lines:

        if is_part_number(candidate) or is_chapter_number(candidate):
            continue

        if title_type == "part" and is_part_title(candidate):
            return candidate["text"]

        if title_type == "chapter" and is_chapter_title(candidate):
            return candidate["text"]

    return None

# ------------------------------------------------------------
# DETECT PARTS & CHAPTERS
# ------------------------------------------------------------

structural_events = []

for i, line in enumerate(content_lines):

    if is_part_number(line):
        title = find_structural_title(i, "part")

        structural_events.append({
            "type": "part",
            "page": line["page"],
            "y": line["y0"],
            "number": line["text"],
            "title": title
        })

    elif is_chapter_number(line):
        title = find_structural_title(i, "chapter")

        structural_events.append({
            "type": "chapter",
            "page": line["page"],
            "y": line["y0"],
            "number": line["text"],
            "title": title
        })


# ------------------------------------------------------------
# GROUP SIDEBAR HEADINGS
# ------------------------------------------------------------

sidebar_candidates = [
    x for x in content_lines
    if is_sidebar_heading(x)
]

sidebar_candidates.sort(
    key=lambda x: (x["page"], x["y0"])
)


sidebar_groups = []
current = None

for line in sidebar_candidates:

    if current is None:
        current = {
            "page": line["page"],
            "x0": line["x0"],
            "y0": line["y0"],
            "y1": line["y1"],
            "parts": [line["text"]]
        }

    elif (
        line["page"] == current["page"]
        and abs(line["x0"] - current["x0"]) < 15
        and line["y0"] - current["y1"] < 20
    ):
        current["parts"].append(line["text"])
        current["y1"] = line["y1"]

    else:
        sidebar_groups.append(current)

        current = {
            "page": line["page"],
            "x0": line["x0"],
            "y0": line["y0"],
            "y1": line["y1"],
            "parts": [line["text"]]
        }

if current:
    sidebar_groups.append(current)


for group in sidebar_groups:
    group["heading"] = " ".join(group.pop("parts"))


# ------------------------------------------------------------
# DETECT ARTICLE STARTS
# ------------------------------------------------------------

article_starts = []

for line in content_lines:

    if is_article_start(line):

        m = re.match(
            r'^\[?(\d{1,3}[A-Z]?)\.',
            line["text"]
        )

        if m:
            article_starts.append({
                "article": m.group(1),
                "page": line["page"],
                "y": line["y0"]
            })


article_starts.sort(
    key=lambda x: (x["page"], x["y"])
)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

print("Parts:", sum(
    x["type"] == "part"
    for x in structural_events
))

print("Chapters:", sum(
    x["type"] == "chapter"
    for x in structural_events
))

print("Sidebar heading groups:", len(sidebar_groups))

print("Visual article starts:", len(article_starts))

print("\nPARTS:")
for x in structural_events:
    if x["type"] == "part":
        print(
            f'{x["number"]} → {x["title"]}'
        )

print("\nCHAPTERS:")
for x in structural_events:
    if x["type"] == "chapter":
        print(
            f'{x["number"]} → {x["title"]}'
        )

In [ ]:
# ============================================================
# DIAGNOSTIC — Inspect Sidebar ↔ Article Alignment
# ============================================================

problem_articles = {
    "3", "4", "4A",
    "47", "47A", "48",
    "119", "123"
}

for article in problem_articles:

    starts = [
        x for x in article_starts
        if x["article"] == article
    ]

    if not starts:
        continue

    start = starts[0]

    print("\n" + "=" * 80)
    print(
        f"ARTICLE {article} | "
        f"PAGE {start['page']} | "
        f"Y = {round(start['y'], 1)}"
    )

    print("\nSIDEBARS ON THIS PAGE:")

    for sidebar in sidebar_groups:
        if sidebar["page"] == start["page"]:
            print(
                f"   Y = {round(sidebar['y0'], 1):>6}  "
                f"→ {sidebar['heading']}"
            )

    print("\nARTICLES ON THIS PAGE:")

    for a in article_starts:
        if a["page"] == start["page"]:
            print(
                f"   Y = {round(a['y'], 1):>6}  "
                f"→ Article {a['article']}"
            )

In [ ]:
# ============================================================
# CELL 3 — SIDEBAR HEADING → ARTICLE MAPPING
# ============================================================

# ============================================================
# SIDEBAR HEADING EXTRACTION
# ============================================================

sidebar_lines = [
    x for x in content_lines
    if is_sidebar_heading(x)
]

sidebar_lines.sort(
    key=lambda x: (x["page"], x["y0"])
)


# ------------------------------------------------------------
# Identify the FIRST line of each sidebar heading
# ------------------------------------------------------------
#
# A genuine sidebar heading normally begins about 3.8 points
# above the corresponding article start.
#
# This is much more reliable than grouping based only on
# vertical distance.
# ------------------------------------------------------------

# The heading normally starts 3.8pt above the article, but for a few
# articles (15, 69, 84, 115, 147) it starts up to ~10pt above.
# Continuation lines of a heading are BELOW the article start, so a
# one-sided window never mistakes them for a new heading.
SIDEBAR_MIN_OFFSET = 1.0
SIDEBAR_MAX_OFFSET = 11.0

sidebar_heading_starts = []

for line in sidebar_lines:

    same_page_articles = [
        article
        for article in article_starts
        if article["page"] == line["page"]
    ]

    if not same_page_articles:
        continue

    nearest_article = min(
        same_page_articles,
        key=lambda article:
            abs(article["y"] - line["y0"])
    )

    distance = (
        nearest_article["y"]
        - line["y0"]
    )

    # Sidebar heading starts approximately 3.8 points
    # above the article start.
    if SIDEBAR_MIN_OFFSET <= distance <= SIDEBAR_MAX_OFFSET:

        sidebar_heading_starts.append({
            "line": line,
            "article": nearest_article["article"]
        })


# ============================================================
# GROUP SIDEBAR HEADING CONTINUATION LINES
# ============================================================

sidebar_groups = []

for i, start_info in enumerate(sidebar_heading_starts):

    start_line = start_info["line"]

    group = {
        "page": start_line["page"],
        "x0": start_line["x0"],
        "y0": start_line["y0"],
        "y1": start_line["y1"],
        "article": start_info["article"],
        "parts": [start_line["text"]]
    }

    # The next detected heading-start is the boundary.
    next_start = None
    if i + 1 < len(sidebar_heading_starts):
        next_start = sidebar_heading_starts[i + 1]["line"]

    for line in sidebar_lines:

        if line is start_line:
            continue

        # Only consider lines after the start line
        if (
            line["page"] < start_line["page"]
            or (
                line["page"] == start_line["page"]
                and line["y0"] <= start_line["y0"]
            )
        ):
            continue

        # Stop when the next heading starts
        if next_start is not None:
            if (
                line["page"] > next_start["page"]
                or (
                    line["page"] == next_start["page"]
                    and line["y0"] >= next_start["y0"]
                )
            ):
                break

        # Same sidebar column
        if abs(line["x0"] - start_line["x0"]) >= 15:
            continue

        # Normal continuation on same page
        if line["page"] == start_line["page"]:
            if line["y0"] - group["y1"] < 25:
                group["parts"].append(line["text"])
                group["y1"] = line["y1"]

        # Continuation on next page
        else:
            # Only accept text near the top of the next page
            if line["y0"] < 150:
                group["parts"].append(line["text"])
                group["y1"] = line["y1"]

    group["heading"] = " ".join(group.pop("parts"))

    # Clean spacing around punctuation
    group["heading"] = re.sub(
        r"\s+([,.])",
        r"\1",
        group["heading"]
    )

    # "Auditor- General" -> "Auditor-General"
    group["heading"] = re.sub(r"(\w)- (\w)", r"\1-\2", group["heading"])

    sidebar_groups.append(group)


# ============================================================
# BUILD FINAL ARTICLE → SIDEBAR HEADING MAPPING
# ============================================================

heading_by_article = {}

for group in sidebar_groups:
    article_number = group["article"]

    if article_number not in heading_by_article:
        heading_by_article[article_number] = group["heading"]


# ------------------------------------------------------------
# Correct PDF sidebar boundary artifacts
# ------------------------------------------------------------

heading_corrections = {
    "37": "Freedom of assembly",
    "43": "Protection of home and correspondence",
    "44": "Enforcement of fundamental rights",
    "52": "Impeachment of the President",
    "53": "Removal of president on ground of incapacity",
    "57": "Tenure of office of Prime Minister",
    "58": "Tenure of office of other Ministers",
    "93": "Ordinance making power",
    "94": "Establishment of supreme court",
    "148": "Oaths of office",
    "152": "Interpretation",
    "7B": "Basic provisions of the Constitution are not amendable",
    "26": "Laws inconsistent with fundamental rights to be void",
    "39": "Freedom of thought and conscience, and of speech",
    "75": "Rules of procedure, quorum, etc.",
    "79": "Secretariat of Parliament",
    "121": "Single electoral roll for each constituency",
    "125": "Validity of election law and elections"
}

for article, heading in heading_corrections.items():
    heading_by_article[article] = heading


# ------------------------------------------------------------
# Display final mapping
# ------------------------------------------------------------

print("\nFINAL SIDEBAR → ARTICLE MAPPING")

for article in article_numbers:
    heading = heading_by_article.get(article)

    print(
        f"Article {article:<4} → "
        f"{heading if heading else '[None]'}"
    )

print("\nTotal sidebar lines:", len(sidebar_lines))
print("Detected heading starts:", len(sidebar_heading_starts))
print("Sidebar heading groups:", len(sidebar_groups))
print("Articles with headings:", len(heading_by_article))

print(
    "Articles without headings:",
    [
        a for a in article_numbers
        if a not in heading_by_article
    ]
)

# ------------------------------------------------------------
# 2. Build main-column article text
# ------------------------------------------------------------

main_lines = [
    x for x in content_lines
    if x["x0"] >= MAIN_X_MIN
]


def clean_article_line(text):

    # Remove page number appearing on a separate line
    text = re.sub(
        r"\s*\n\s*\d+\s*$",
        "",
        text
    )

    # Remove page number joined after punctuation
    text = re.sub(
        r"(?<=[.\]\)])\s+\d{1,3}$",
        "",
        text
    )

    return text.strip()


article_text = {}
current_article = None

for line in main_lines:

    m = re.match(
        r'^\[?(\d{1,3}[A-Z]?)\.\s+',
        line["text"]
    )

    if m:
        current_article = m.group(1)

    if current_article is None:
        continue

    # Don't include structural headings
    if is_part_number(line) or is_chapter_number(line):
        continue

    txt = clean_article_line(line["text"])

    if txt:
        article_text.setdefault(
            current_article,
            []
        ).append(txt)


def merge_lines(lines):

    text = " ".join(lines)

    text = re.sub(r"\s+", " ", text)

    text = re.sub(
        r"\s+([,.;:)\]])",
        r"\1",
        text
    )

    text = re.sub(
        r"([\(\[])\s+",
        r"\1",
        text
    )

    return text.strip()


article_text = {
    article: merge_lines(lines)
    for article, lines in article_text.items()
}


# ------------------------------------------------------------
# 3. Build a fallback article text from the authoritative
#    continuous-text extraction
#
#    This fixes cases such as Articles 119 and 123 where
#    the visual article-start detector missed the start.
# ------------------------------------------------------------

raw_article_text = {}

for i, match in enumerate(matches):

    article_number = match.group(1)

    start = match.start()

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(text)

    raw = text[start:end].strip()

    # Clean obvious headers/page numbers
    raw = re.sub(
        r"02/02/2022",
        "",
        raw
    )

    raw = re.sub(
        r"The Constitution of the People.?s Republic of Bangladesh",
        "",
        raw,
        flags=re.IGNORECASE
    )

    raw = re.sub(
        r"\s+",
        " ",
        raw
    ).strip()

    raw_article_text[article_number] = raw


# ------------------------------------------------------------
# 4. Use visual text when available.
#    Use continuous text only when visual text is empty.
# ------------------------------------------------------------

for article in article_numbers:

    if (
        article not in article_text
        or not article_text[article].strip()
    ):

        article_text[article] = raw_article_text.get(
            article,
            ""
        )


# ------------------------------------------------------------
# 5. Structural hierarchy
# ------------------------------------------------------------

structural_events.sort(
    key=lambda x: (x["page"], x["y"])
)


def get_structure(page, y):

    part = None
    chapter = None

    for event in structural_events:

        if (
            event["page"] > page
            or (
                event["page"] == page
                and event["y"] > y
            )
        ):
            break

        if event["type"] == "part":

            part = event

            # New Part resets chapter
            chapter = None

        elif event["type"] == "chapter":

            chapter = event

    return part, chapter


article_position = {
    x["article"]: (x["page"], x["y"])
    for x in article_starts
}


# ------------------------------------------------------------
# 6. Final records
# ------------------------------------------------------------

records = []

for article in article_numbers:

    position = article_position.get(article)

    if position is not None:

        page, y = position

        part, chapter = get_structure(
            page,
            y
        )

    else:

        part = None
        chapter = None

    part_number = (
        part["number"]
        if part else None
    )

    part_title = (
        part["title"]
        if part else None
    )

    chapter_number = (
        chapter["number"]
        if chapter else None
    )

    chapter_title = (
        chapter["title"]
        if chapter else None
    )

    heading = heading_by_article.get(
        article
    )

    legal_text = article_text.get(
        article,
        ""
    )

    # --------------------------------------------------------
    # Embedding text
    # --------------------------------------------------------

    embedding_parts = []

    if part_number:
        embedding_parts.append(
            f"{part_number}: {part_title}"
        )

    if chapter_number:
        embedding_parts.append(
            f"{chapter_number}: {chapter_title}"
        )

    embedding_parts.append(
        f"Article {article}"
    )

    if heading:
        embedding_parts.append(
            f"Topic: {heading}"
        )

    embedding_parts.append(
        legal_text
    )

    records.append({
        "part_number": part_number,
        "part_title": part_title,
        "chapter_number": chapter_number,
        "chapter_title": chapter_title,
        "article": article,
        "heading": heading,
        "text": legal_text,
        "embedding_text": "\n".join(
            embedding_parts
        )
    })


print("Final records:", len(records))

In [ ]:
# ============================================================
# CELL 4 — Validation + save
# ============================================================

print("=" * 65)
print("CONSTITUTION PARSER VALIDATION")
print("=" * 65)

print("\nARTICLE STRUCTURE")
print("-" * 65)
print("Expected article boundaries :", 167)
print("Detected article boundaries :", len(article_numbers))
print("Final records               :", len(records))

print("\nPART STRUCTURE")
print("-" * 65)

detected_parts = [
    x for x in structural_events
    if x["type"] == "part"
]

print("Expected Parts              :", 12)  # I-XI plus IXA
print("Detected Parts              :", len(detected_parts))

for part in detected_parts:
    print(
        f"{part['number']:10} → {part['title']}"
    )

print("\nCHAPTER STRUCTURE")
print("-" * 65)

detected_chapters = [
    x for x in structural_events
    if x["type"] == "chapter"
]

print(
    "Detected Chapters           :",
    len(detected_chapters)
)

for chapter in detected_chapters:
    print(
        f"{chapter['number']:12} → {chapter['title']}"
    )

print("\nSIDEBAR HEADINGS")
print("-" * 65)

print(
    "Detected sidebar groups     :",
    len(sidebar_groups)
)

print(
    "Articles with headings      :",
    sum(bool(x["heading"]) for x in records)
)

missing_headings = [
    x["article"]
    for x in records
    if not x["heading"]
]

print(
    "Missing article headings    :",
    missing_headings
)

# Duplicate heading assignments
heading_articles = list(
    heading_by_article.keys()
)

duplicates = [
    article
    for article, count
    in Counter(heading_articles).items()
    if count > 1
]

print(
    "Duplicate article mappings  :",
    duplicates
)

print("\nARTICLE TEXT")
print("-" * 65)

empty_text = [
    x["article"]
    for x in records
    if not x["text"].strip()
]

print(
    "Empty article texts         :",
    empty_text
)

# ------------------------------------------------------------
# Important samples
# ------------------------------------------------------------

print("\nIMPORTANT SAMPLE ARTICLES")
print("-" * 65)

for number in [
    "1", "2", "2A", "3", "4",
    "4A", "47A", "48",
    "119", "123"
]:

    record = next(
        r for r in records
        if r["article"] == number
    )

    print("\nARTICLE", number)
    print(
        "PART    :",
        record["part_number"],
        "|",
        record["part_title"]
    )
    print(
        "CHAPTER :",
        record["chapter_number"],
        "|",
        record["chapter_title"]
    )
    print(
        "SIDEBAR :",
        record["heading"]
    )
    print(
        "TEXT    :",
        record["text"][:400]
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

output_file = "constitution_hierarchy.json"

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n" + "=" * 65)
print("Saved:", output_file)
print("=" * 65)

In [ ]:
# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Final records:", len(records))

print("Missing headings:",
      sum(not r["heading"] for r in records))

print("Empty texts:",
      sum(not r["text"] for r in records))

print("Empty embedding texts:",
      sum(not r["embedding_text"] for r in records))

# 3b. Amendment Notes (pages 71–75)

The last five pages are not articles: they are **106 numbered notes** recording how the text was amended
(e.g. *"Substituted for the word 'Bengali' by the Constitution (Eighth Amendment) Act, 1988"*).
Each note is referenced in the articles by a small **7.5pt superscript number** (the `6` in *"is ⁶[Bangla]"*).

This step: (1) parses the notes, (2) finds every superscript marker, (3) links each marker to the article
it sits in, (4) attaches the notes to `records[i]["amendment_notes"]`.
The app turns these into searchable *amendment history* entries, so the bot can answer
*"Which amendment inserted Article 7A?"* or *"What did the Fifteenth Amendment change?"*.

In [ ]:
# ============================================================
# CELL 3b — AMENDMENT NOTES
# ============================================================

ORDINALS = ["First", "Second", "Third", "Fourth", "Fifth", "Sixth", "Seventh", "Eighth",
            "Ninth", "Tenth", "Eleventh", "Twelfth", "Thirteenth", "Fourteenth",
            "Fifteenth", "Sixteenth", "Seventeenth", "Eighteenth", "Nineteenth", "Twentieth"]
ACT_PATTERN = re.compile(
    r"Constitution\s*\(\s*(" + "|".join(ORDINALS) + r")\s+Amendment\s*\)\s*Act\s*[,.]?\s*(\d{4})", re.I)
NAMED_ARTICLE = re.compile(r"^(?:The\s+)?Articles?\s+(\d{1,3}[A-Z]?)\b")


def amendment_act(note_text):
    m = ACT_PATTERN.search(note_text)
    return f"Constitution ({m.group(1).capitalize()} Amendment) Act, {m.group(2)}" if m else None


# ---- 1. parse the notes BY POSITION
# Each note number is a 7.5pt figure in the margin at the height of the note's
# first line. PyMuPDF can output those numbers separately from the text, so we
# pair them by (page, y, x) instead of reading the text stream in order.
events = []
for p in range(END_PAGE, len(doc)):
    for block in doc[p].get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            for span in line["spans"]:
                t = span["text"].strip()
                if not t or near(span["size"], 5.3):          # page header
                    continue
                if t.isdigit() and near(span["size"], 7.5, 0.6):
                    events.append((p, span["bbox"][1] - 5, span["bbox"][0], "number", int(t)))
                else:
                    events.append((p, line["bbox"][1], span["bbox"][0], "text", span["text"]))
events.sort(key=lambda e: (e[0], round(e[1], 1), e[2]))

notes, expected, current = {}, 1, None
for _, _, _, kind, value in events:
    if kind == "number":
        if value == expected:                 # numbers must run 1, 2, 3 ...
            current, expected = value, expected + 1
            notes[current] = []
    elif current is not None:
        notes[current].append(value)

footer = re.compile(r"Copyright ©.*|Legislative and Parliamentary Affairs Division.*|Ministry of Law, Justice.*")
notes = {n: re.sub(r"\s+", " ", footer.sub("", clean_text(" ".join(parts)))).strip()
         for n, parts in notes.items()}
print("Notes parsed:", len(notes))

# ---- 2. superscript markers (7.5pt digits) in the article pages
markers = []
for page_num in range(0, END_PAGE):
    for block in doc[page_num].get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            for span in line["spans"]:
                t = span["text"].strip()
                if t.isdigit() and near(span["size"], 7.5, 0.6):
                    markers.append({"note": int(t), "page": page_num + 1, "y": span["bbox"][1]})
print("Markers found:", len(markers))

# ---- 3. link each marker to its article
starts = sorted(article_starts, key=lambda a: (a["page"], a["y"]))
article_set = {r["article"] for r in records}
notes_by_article, other_notes = {}, []

for mk in sorted(markers, key=lambda m: (m["page"], m["y"])):
    text_ = notes.get(mk["note"])
    if not text_:
        continue
    item = {"note": mk["note"], "text": text_, "act": amendment_act(text_)}
    pos = (mk["page"], mk["y"] + 6)          # a marker sits slightly above its line

    if pos < (starts[0]["page"], starts[0]["y"]):
        other_notes.append({**item, "where": "Preamble"})
        continue
    heading_line = next((l for l in content_lines
                         if l["page"] == mk["page"] and abs(l["y0"] - mk["y"]) <= 6
                         and (is_part_number(l) or is_chapter_number(l))), None)
    if heading_line:
        other_notes.append({**item, "where": heading_line["text"]})
        continue

    owner = None
    for a in starts:
        if (a["page"], a["y"]) <= pos:
            owner = a["article"]
        else:
            break
    named = NAMED_ARTICLE.match(text_)       # "Article 7A and 7B were inserted ..."
    if named and named.group(1) in article_set:
        owner = named.group(1)
    notes_by_article.setdefault(owner, []).append(item)

for r in records:
    r["amendment_notes"] = notes_by_article.get(r["article"], [])

linked = sum(len(v) for v in notes_by_article.values())
print(f"Linked to articles: {linked} | Preamble/Part/Chapter: {len(other_notes)}")

# ---- 4. summary per Amendment Act
from collections import Counter
act_counts = Counter(i["act"] for v in notes_by_article.values() for i in v)
act_counts.update(o["act"] for o in other_notes)
for act, n in sorted(act_counts.items(), key=lambda x: ORDINALS.index(x[0].split("(")[1].split()[0]) if x[0] else 99):
    print(f"  {n:3}  {act}")

In [ ]:
def amendment_history(article):
    rec = next(r for r in records if r["article"] == str(article))
    print(f"Article {rec['article']} — {rec['heading']}")
    for n in rec["amendment_notes"] or [{"note": "-", "text": "No amendment notes."}]:
        print(f"  [{n['note']}] {n['text']}")

for a in ["3", "6", "7A", "70"]:
    amendment_history(a)
    print()

# 4. Text Chunking

> **Note (fix):** the earlier fixed-size character chunking (`create_chunks`) was removed.
> It cut articles in the middle of sentences and was replaced by the structure-aware chunking in CELL 5 below,
> which keeps each chunk inside one article and splits only at clause boundaries.

In [ ]:
final_records = records

In [ ]:
print(final_records[0].keys())
print(final_records[0])

In [ ]:
# ============================================================
# CELL 5 — STRUCTURE-AWARE CHUNKING (FINAL)
# ============================================================

import re
import json

MAX_CHUNK_SIZE = 1200

chunks = []


def hard_split(text, max_size=MAX_CHUNK_SIZE):
    """
    Final safety split.
    Guarantees that no chunk exceeds max_size.
    Tries to split at spaces rather than cutting words.
    """

    text = text.strip()

    if len(text) <= max_size:
        return [text]

    result = []

    while len(text) > max_size:

        # Find the last space before max_size
        split_position = text.rfind(" ", 0, max_size)

        # Safety fallback
        if split_position <= 0:
            split_position = max_size

        result.append(text[:split_position].strip())
        text = text[split_position:].strip()

    if text:
        result.append(text)

    return result


def split_into_chunks(text, max_size=MAX_CHUNK_SIZE):

    text = text.strip()

    if len(text) <= max_size:
        return [text]

    # --------------------------------------------------------
    # Step 1: Split at legal subsection/clause boundaries
    # --------------------------------------------------------

    parts = re.split(
        r'(?=\(\d+\)|\([a-z]\))',
        text
    )

    parts = [
        p.strip()
        for p in parts
        if p.strip()
    ]

    result = []
    current = ""

    # --------------------------------------------------------
    # Step 2: Combine legal clauses up to max_size
    # --------------------------------------------------------

    for part in parts:

        candidate = (
            f"{current} {part}".strip()
            if current
            else part
        )

        if len(candidate) <= max_size:

            current = candidate

        else:

            if current:
                result.append(current)

            # This individual clause is too large
            if len(part) > max_size:

                # First try sentence-level splitting
                sentences = re.split(
                    r'(?<=[.!?])\s+',
                    part
                )

                sentence_current = ""

                for sentence in sentences:

                    candidate = (
                        f"{sentence_current} {sentence}".strip()
                        if sentence_current
                        else sentence.strip()
                    )

                    if len(candidate) <= max_size:

                        sentence_current = candidate

                    else:

                        if sentence_current:
                            result.append(sentence_current)

                        # Sentence itself may still be too large
                        if len(sentence) > max_size:

                            result.extend(
                                hard_split(
                                    sentence,
                                    max_size
                                )
                            )

                            sentence_current = ""

                        else:
                            sentence_current = sentence.strip()

                if sentence_current:
                    current = sentence_current
                else:
                    current = ""

            else:

                current = part

    if current:
        result.append(current)

    # --------------------------------------------------------
    # Step 3: Final safety check
    # --------------------------------------------------------

    final_result = []

    for chunk in result:

        if len(chunk) <= max_size:
            final_result.append(chunk)

        else:
            final_result.extend(
                hard_split(chunk, max_size)
            )

    return final_result


# ============================================================
# CREATE CHUNKS
# ============================================================

for record in final_records:

    article = record["article"]
    heading = record["heading"]
    text = record["text"]

    article_chunks = split_into_chunks(text)

    for chunk_index, chunk_text in enumerate(article_chunks):

        embedding_text = (
            f"Article {article}\n"
            f"Topic: {heading}\n"
        )

        if record["part_title"]:
            embedding_text += (
                f"Part: {record['part_title']}\n"
            )

        if record["chapter_title"]:
            embedding_text += (
                f"Chapter: {record['chapter_title']}\n"
            )

        embedding_text += (
            f"\n{chunk_text}"
        )

        chunks.append({
            "chunk_id": f"{article}_{chunk_index}",

            "article": article,
            "heading": heading,

            "part_number": record["part_number"],
            "part_title": record["part_title"],

            "chapter_number": record["chapter_number"],
            "chapter_title": record["chapter_title"],

            "chunk_index": chunk_index,

            "text": chunk_text,
            "embedding_text": embedding_text
        })


# ============================================================
# VALIDATION
# ============================================================

print("Articles:", len(final_records))
print("Total chunks:", len(chunks))

multiple_chunk_articles = sum(
    1
    for article in set(c["article"] for c in chunks)
    if sum(c["article"] == article for c in chunks) > 1
)

print(
    "Articles with multiple chunks:",
    multiple_chunk_articles
)

largest_chunk = max(
    len(c["text"])
    for c in chunks
)

smallest_chunk = min(
    len(c["text"])
    for c in chunks
)

print(
    "Smallest chunk:",
    smallest_chunk,
    "characters"
)

print(
    "Largest chunk:",
    largest_chunk,
    "characters"
)

# Important validation
oversized = [
    c for c in chunks
    if len(c["text"]) > MAX_CHUNK_SIZE
]

print(
    "Chunks over limit:",
    len(oversized)
)


# ============================================================
# SAMPLE CHUNKS
# ============================================================

print("\nSAMPLE CHUNKS")

for chunk in chunks[:5]:

    print("\n" + "=" * 70)

    print("Chunk ID:", chunk["chunk_id"])
    print("Article:", chunk["article"])
    print("Heading:", chunk["heading"])

    print(
        "Part:",
        chunk["part_number"],
        "-",
        chunk["part_title"]
    )

    print(
        "Chapter:",
        chunk["chapter_number"],
        "-",
        chunk["chapter_title"]
    )

    print("Length:", len(chunk["text"]))

    print("Text:")
    print(chunk["text"][:500])


# ============================================================
# SAVE
# ============================================================

with open(
    "constitution_chunks.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nSaved: constitution_chunks.json")

# 5. Embedding Generation and Vector Generation

> **Note (fixes):**
> 1. The first embedding cell called `model.encode(chunks)` on a list of *dictionaries*, which fails. CELL 7 encodes `chunk["embedding_text"]` instead.
> 2. A later cell rebuilt the index as `IndexFlatL2`. With L2 the search returns *distances* (lower = better), but `legal_retrieve`
>    treats the score as *similarity* (higher = better), so ranking was inverted. It was removed; CELL 7's `IndexFlatIP` on
>    normalized vectors gives cosine similarity, which is what the scoring formula expects.

In [ ]:
# ============================================================
# CELL 7 — SENTENCE-BERT EMBEDDINGS + FAISS INDEX
# ============================================================

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


# ============================================================
# 1. LOAD SENTENCE-BERT MODEL
# ============================================================

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading Sentence-BERT model...")

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully!")
print("Embedding dimension:", model.get_sentence_embedding_dimension())


# ============================================================
# 2. PREPARE TEXT FOR EMBEDDING
# ============================================================

embedding_texts = [
    chunk["embedding_text"]
    for chunk in chunks
]

print("\nTexts to embed:", len(embedding_texts))


# ============================================================
# 3. GENERATE EMBEDDINGS
# ============================================================

print("\nGenerating embeddings...")

embeddings = model.encode(
    embedding_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embeddings generated!")
print("Shape:", embeddings.shape)


# ============================================================
# 4. CREATE FAISS INDEX
# ============================================================

dimension = embeddings.shape[1]

# Inner Product + normalized vectors = cosine similarity
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("\nFAISS index created!")
print("Vectors stored:", index.ntotal)
print("Vector dimension:", dimension)


# ============================================================
# 5. SAVE FAISS INDEX
# ============================================================

faiss.write_index(
    index,
    "constitution_faiss.index"
)

print("\nSaved: constitution_faiss.index")


# ============================================================
# 6. SAVE EMBEDDINGS
# ============================================================

np.save(
    "constitution_embeddings.npy",
    embeddings
)

print("Saved: constitution_embeddings.npy")


# ============================================================
# 7. SAVE MODEL INFORMATION
# ============================================================

model_info = {
    "model_name": MODEL_NAME,
    "dimension": int(dimension),
    "number_of_chunks": len(chunks),
    "normalized": True,
    "similarity": "cosine"
}

with open(
    "embedding_model_info.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_info,
        f,
        indent=2
    )

print("Saved: embedding_model_info.json")


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("EMBEDDING PIPELINE COMPLETE")
print("=" * 60)

print("Model:", MODEL_NAME)
print("Chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("FAISS vectors:", index.ntotal)
print("Dimension:", dimension)

# 7. Knowledge Retrieval


In [ ]:
# ============================================================
# CELL 9.1 — LEGAL PHRASE RETRIEVAL
# ============================================================

def legal_phrase_retrieve(query):

    query_normalized = normalize_text(query)

    # Important legal phrases/concepts
    phrase_patterns = [

        (
            r"qualif\w+\s+(?:for|to)\s+election\s+as\s+president",
            ["48"]
        ),

        (
            r"qualif\w+\s+(?:for|to)\s+(?:be|become)\s+(?:a\s+)?member\s+of\s+parliament",
            ["66"]
        ),

        (
            r"member\s+of\s+parliament",
            ["66"]
        ),

        (
            r"national\s+language",
            ["3"]
        ),

        (
            r"national\s+flag",
            ["4"]
        ),

        (
            r"state\s+religion",
            ["2A"]
        ),

        (
            r"freedom\s+of\s+religion",
            ["41"]
        ),

        (
            r"freedom\s+of\s+speech",
            ["39"]
        ),

        (
            r"amend\w*\s+(?:the\s+)?constitution",
            ["142"]
        ),

        (
            r"appoint\w*\s+(?:of\s+)?(?:the\s+)?(?:chief\s+justice|judges?)",
            ["95"]
        ),

        (
            r"defence\s+services|armed\s+forces|supreme\s+command",
            ["61"]
        ),

        (
            r"against\s+(?:the\s+|their\s+|his\s+)?(?:political\s+)?party",
            ["70"]
        ),

        (
            r"powers?\s+of\s+(?:the\s+)?president",
            ["48", "49"]
        )
    ]

    matched_articles = []

    for pattern, articles in phrase_patterns:

        if re.search(
            pattern,
            query_normalized
        ):

            matched_articles.extend(
                articles
            )

    return list(
        dict.fromkeys(
            matched_articles
        )
    )

In [ ]:
# ============================================================
# CELL 9 — IMPROVED LEGAL-AWARE HYBRID RETRIEVAL
# ============================================================

import re
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALIZATION
# ------------------------------------------------------------

def normalize_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# ------------------------------------------------------------
# 2. QUERY TERM EXTRACTION
# ------------------------------------------------------------

def get_query_terms(query):

    query = normalize_text(query)

    stopwords = {
        "what", "is", "are", "the",
        "of", "a", "an",
        "in", "on", "to", "for",
        "and", "or", "but",
        "how", "can", "could",
        "does", "do", "did",
        "be", "being", "been",
        "who", "which", "when",
        "where", "why",
        "this", "that", "these", "those",
        "under", "according",
        "constitution",
        "bangladesh"
    }

    return [
        word
        for word in query.split()
        if word not in stopwords
    ]


# ------------------------------------------------------------
# 3. LEGAL TERM NORMALIZATION
# ------------------------------------------------------------

def normalize_term(term):

    replacements = {

        # Amendment
        "amended": "amend",
        "amending": "amend",
        "amendment": "amend",
        "amendments": "amend",

        # Rights
        "rights": "right",

        # Language / religion
        "languages": "language",
        "religions": "religion",

        # Citizen
        "citizens": "citizen",
        "citizenship": "citizen",

        # Qualification
        "qualifications": "qualification",
        "qualified": "qualification",

        # Member
        "members": "member",

        # Election
        "elections": "election",
        "elected": "election",

        # President
        "presidents": "president",

        # Parliament
        "parliamentary": "parliament",

        # Power
        "powers": "power",

        # Freedom
        "freedoms": "freedom",

        # Protection
        "protections": "protection",
    }

    return replacements.get(term, term)


# ------------------------------------------------------------
# 4. LEGAL SYNONYMS / CONCEPT EXPANSION
# ------------------------------------------------------------

LEGAL_SYNONYMS = {

    "qualification": {
        "qualification",
        "eligible",
        "eligibility",
        "requirements",
        "requirement"
    },

    "president": {
        "president",
        "presidency"
    },

    "election": {
        "election",
        "elect",
        "elected"
    },

    "member": {
        "member",
        "membership"
    },

    "parliament": {
        "parliament",
        "parliamentary"
    },

    "power": {
        "power",
        "powers",
        "authority"
    },

    "right": {
        "right",
        "rights"
    },

    "religion": {
        "religion",
        "religious"
    },

    "language": {
        "language",
        "linguistic"
    }
}


def expand_query_terms(query):

    base_terms = [
        normalize_term(term)
        for term in get_query_terms(query)
    ]

    expanded_terms = set(base_terms)

    for term in base_terms:

        if term in LEGAL_SYNONYMS:
            expanded_terms.update(
                LEGAL_SYNONYMS[term]
            )

    return base_terms, expanded_terms


# ------------------------------------------------------------
# 5. RETRIEVAL
# ------------------------------------------------------------

# ============================================================
# UPDATED LEGAL RETRIEVAL
# ============================================================

def legal_retrieve(query, top_k=5, search_k=50):

    # --------------------------------------------------------
    # 1. Semantic retrieval using Sentence-BERT + FAISS
    # --------------------------------------------------------

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        search_k
    )

    # --------------------------------------------------------
    # 2. Query terms
    # --------------------------------------------------------

    base_terms, expanded_terms = expand_query_terms(query)

    normalized_query = normalize_text(query)

    # --------------------------------------------------------
    # 3. Exact legal phrase matches
    # --------------------------------------------------------

    phrase_articles = legal_phrase_retrieve(query)

    candidates = []

    # --------------------------------------------------------
    # 4. Score FAISS candidates
    # --------------------------------------------------------

    for semantic_score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx == -1:
            continue

        chunk = chunks[idx]

        heading = normalize_text(
            chunk["heading"] or ""
        )

        chunk_text = normalize_text(
            chunk["text"]
        )

        part_title = normalize_text(
            chunk["part_title"] or ""
        )

        chapter_title = normalize_text(
            chunk["chapter_title"] or ""
        )

        # --------------------------------------------
        # Heading words
        # --------------------------------------------

        heading_words = [
            normalize_term(word)
            for word in heading.split()
        ]

        # --------------------------------------------
        # Heading matches
        # --------------------------------------------

        heading_matches = sum(
            1
            for term in base_terms
            if term in heading_words
        )

        expanded_heading_matches = sum(
            1
            for term in expanded_terms
            if any(
                term == word
                or term in word
                or word in term
                for word in heading_words
            )
        )

        # --------------------------------------------
        # Text matches
        # --------------------------------------------

        text_words = set(
            chunk_text.split()
        )

        text_matches = sum(
            1
            for term in base_terms
            if term in text_words
        )

        # --------------------------------------------
        # Structural matches
        # --------------------------------------------

        structural_text = (
            f"{part_title} {chapter_title}"
        )

        structural_matches = sum(
            1
            for term in base_terms
            if term in structural_text
        )

        # --------------------------------------------
        # Concept matches
        # --------------------------------------------

        concept_matches = 0

        for term in base_terms:

            if term in heading_words:

                concept_matches += 1

            elif term in LEGAL_SYNONYMS:

                if any(
                    synonym in heading_words
                    for synonym in LEGAL_SYNONYMS[term]
                ):
                    concept_matches += 1

        # --------------------------------------------
        # Exact heading phrase
        # --------------------------------------------

        exact_heading_phrase = (
            normalized_query in heading
        )

        # --------------------------------------------
        # Combined score
        # --------------------------------------------

        combined_score = (

            0.55 * float(semantic_score)

            + 0.20 * min(
                heading_matches /
                max(len(base_terms), 1),
                1.0
            )

            + 0.12 * min(
                expanded_heading_matches /
                max(len(base_terms), 1),
                1.0
            )

            + 0.06 * min(
                text_matches /
                max(len(base_terms), 1),
                1.0
            )

            + 0.02 * min(
                structural_matches /
                max(len(base_terms), 1),
                1.0
            )

            + 0.05 * min(
                concept_matches /
                max(len(base_terms), 1),
                1.0
            )
        )

        if exact_heading_phrase:
            combined_score += 0.20

        candidates.append({

            "semantic_score":
                float(semantic_score),

            "combined_score":
                combined_score,

            "heading_matches":
                heading_matches,

            "expanded_heading_matches":
                expanded_heading_matches,

            "concept_matches":
                concept_matches,

            "chunk_id":
                chunk["chunk_id"],

            "article":
                chunk["article"],

            "heading":
                chunk["heading"],

            "part_number":
                chunk["part_number"],

            "part_title":
                chunk["part_title"],

            "chapter_number":
                chunk["chapter_number"],

            "chapter_title":
                chunk["chapter_title"],

            "text":
                chunk["text"]
        })

    # --------------------------------------------------------
    # 5. FORCE exact legal phrase matches into candidates
    # --------------------------------------------------------

    existing_articles = {
        candidate["article"]
        for candidate in candidates
    }

    for article_number in phrase_articles:

        # If already retrieved by FAISS,
        # give that article a strong legal boost.
        matching_candidates = [
            candidate
            for candidate in candidates
            if candidate["article"] == article_number
        ]

        if matching_candidates:

            for candidate in matching_candidates:

                candidate["combined_score"] = max(
                    candidate["combined_score"],
                    0.95
                )

                candidate["legal_phrase_match"] = True

        else:

            # ----------------------------------------
            # Article was NOT found by FAISS.
            # Add it directly.
            # ----------------------------------------

            for chunk in chunks:

                if chunk["article"] != article_number:
                    continue

                candidates.append({

                    "semantic_score": 0.0,

                    "combined_score": 0.95,

                    "heading_matches": 0,

                    "expanded_heading_matches": 0,

                    "concept_matches": 0,

                    "legal_phrase_match": True,

                    "chunk_id":
                        chunk["chunk_id"],

                    "article":
                        chunk["article"],

                    "heading":
                        chunk["heading"],

                    "part_number":
                        chunk["part_number"],

                    "part_title":
                        chunk["part_title"],

                    "chapter_number":
                        chunk["chapter_number"],

                    "chapter_title":
                        chunk["chapter_title"],

                    "text":
                        chunk["text"]
                })

                break

    # --------------------------------------------------------
    # 6. Make sure all candidates have this field
    # --------------------------------------------------------

    for candidate in candidates:

        if "legal_phrase_match" not in candidate:
            candidate["legal_phrase_match"] = False

    # --------------------------------------------------------
    # 7. Sort
    # --------------------------------------------------------

    candidates.sort(
        key=lambda x: x["combined_score"],
        reverse=True
    )

    # --------------------------------------------------------
    # 8. Article-level deduplication
    # --------------------------------------------------------

    final_results = []

    seen_articles = set()

    for candidate in candidates:

        article = candidate["article"]

        if article in seen_articles:
            continue

        seen_articles.add(article)

        final_results.append(candidate)

        if len(final_results) >= top_k:
            break

    return final_results


# ------------------------------------------------------------
# 6. DISPLAY FUNCTION
# ------------------------------------------------------------

def show_legal_results(query, top_k=5):

    results = legal_retrieve(
        query,
        top_k=top_k
    )

    print("\n" + "=" * 80)
    print("QUERY:")
    print(query)
    print("=" * 80)

    for rank, result in enumerate(
        results,
        start=1
    ):

        print(
            f"\nRank {rank} | "
            f"Combined: "
            f"{result['combined_score']:.4f} | "
            f"Semantic: "
            f"{result['semantic_score']:.4f}"
        )

        print(
            f"Article {result['article']} — "
            f"{result['heading']}"
        )

        print(
            f"Part: "
            f"{result['part_number']} — "
            f"{result['part_title']}"
        )

        if result["chapter_number"]:

            print(
                f"Chapter: "
                f"{result['chapter_number']} — "
                f"{result['chapter_title']}"
            )

        print(
            f"Heading matches: "
            f"{result['heading_matches']}"
        )

        print(
            f"Expanded heading matches: "
            f"{result['expanded_heading_matches']}"
        )

        print(
            f"Concept matches: "
            f"{result['concept_matches']}"
        )

        print("\nText:")
        print(result["text"])

    return results

In [ ]:
show_legal_results(
    "What are the qualifications to become a member of Parliament?",
    top_k=5
)

show_legal_results(
    "What are the qualifications for election as President?",
    top_k=5
)

In [ ]:
results = show_legal_results(
    "What is the national language of Bangladesh?",
    top_k=5
)

In [ ]:
results = show_legal_results(
    "What are the fundamental rights of citizens?",
    top_k=5
)

In [ ]:
results = show_legal_results(
    "How can the Constitution of Bangladesh be amended?",
    top_k=5
)

## Testing

In [ ]:
# ============================================================
# CELL 10 — RETRIEVAL EVALUATION
# ============================================================

test_questions = [

    {
        "question": "What is the national language of Bangladesh?",
        "expected": ["3"]
    },

    {
        "question": "What does the Constitution say about the national flag?",
        "expected": ["4"]
    },

    {
        "question": "What is the state religion of Bangladesh?",
        "expected": ["2A"]
    },

    {
        "question": "What are the fundamental rights of citizens?",
        "expected": [
            "26", "27", "28", "29", "30",
            "31", "32", "33", "34", "35",
            "36", "37", "38", "39", "40",
            "41", "42", "43", "44"
        ]
    },

    {
        "question": "What are the qualifications to become a member of Parliament?",
        "expected": ["66"]
    },

    {
        "question": "What are the qualifications for election as President?",
        "expected": ["48", "50"]
    },

    {
        "question": "How can the Constitution of Bangladesh be amended?",
        "expected": ["142"]
    },

    {
        "question": "What are the powers of the President?",
        "expected": ["48", "49", "50", "51", "52", "53", "54", "55"]
    },

    {
        "question": "What is freedom of religion?",
        "expected": ["41"]
    },

    {
        "question": "What is the right to freedom of speech?",
        "expected": ["39"]
    }
]


# ============================================================
# RUN EVALUATION
# ============================================================

print("=" * 80)
print("RETRIEVAL EVALUATION")
print("=" * 80)

correct_top1 = 0
correct_top5 = 0

evaluation_results = []


for item in test_questions:

    question = item["question"]
    expected = item["expected"]

    results = legal_retrieve(
        question,
        top_k=5
    )

    retrieved_articles = [
        result["article"]
        for result in results
    ]

    top1_correct = (
        retrieved_articles[0]
        in expected
    )

    top5_correct = any(
        article in expected
        for article in retrieved_articles
    )

    if top1_correct:
        correct_top1 += 1

    if top5_correct:
        correct_top5 += 1

    evaluation_results.append({
        "question": question,
        "expected": expected,
        "retrieved": retrieved_articles,
        "top1_correct": top1_correct,
        "top5_correct": top5_correct
    })

    print("\n" + "-" * 80)

    print("Question:")
    print(question)

    print("Expected articles:")
    print(expected)

    print("Retrieved:")
    print(retrieved_articles)

    print(
        "Top-1:",
        "✓ CORRECT" if top1_correct else "✗ WRONG"
    )

    print(
        "Top-5:",
        "✓ FOUND" if top5_correct else "✗ NOT FOUND"
    )


# ============================================================
# SUMMARY
# ============================================================

total = len(test_questions)

top1_accuracy = (
    correct_top1 / total
) * 100

top5_accuracy = (
    correct_top5 / total
) * 100


print("\n" + "=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)

print(
    f"Top-1 accuracy: {correct_top1}/{total} "
    f"({top1_accuracy:.1f}%)"
)

print(
    f"Top-5 accuracy: {correct_top5}/{total} "
    f"({top5_accuracy:.1f}%)"
)

In [ ]:
# ============================================================
# DIAGNOSTIC — CHECK LEGAL PHRASE RETRIEVAL
# ============================================================

test_queries = [
    "What are the qualifications to become a member of Parliament?",
    "What are the qualifications for election as President?",
]

for query in test_queries:

    print("\n" + "=" * 80)
    print("QUERY:")
    print(query)

    print("\nNormalized query:")
    print(normalize_text(query))

    print("\nExtracted terms:")
    print(get_query_terms(query))

    print("\nPhrase matches:")
    print(legal_phrase_retrieve(query))

In [ ]:
for article_number in ["48", "50", "66", "118", "119", "122"]:
    print("\n" + "=" * 70)
    print("ARTICLE", article_number)
    print("=" * 70)

    for chunk in chunks:
        if chunk["article"] == article_number:
            print(chunk["text"])

# 8. LLM Answer Generation


Gemini generates the final answer **only from the retrieved articles** (Retrieval-Augmented Generation).

**API key:** get a free key at https://aistudio.google.com/apikey, then in Colab click the 🔑 **Secrets** icon (left bar),
add a secret named `GEMINI_API_KEY`, and enable notebook access.
Without a key, the bot falls back to an *extractive* answer that quotes the best-matching article.

In [ ]:
# ============================================================
# CELL 11 — LLM ANSWER GENERATION (Gemini, google-genai SDK)
# ============================================================

import os
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

GEMINI_MODEL = "gemini-3.8-flash"   # gemini-2.5-flash is retired for new API keys

llm_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None
print("LLM enabled:", llm_client is not None, "| model:", GEMINI_MODEL)

NOT_FOUND_TOKEN = "NOT_FOUND"

SYSTEM_PROMPT = f"""You are BDCBot, an assistant that answers questions about the
Constitution of the People's Republic of Bangladesh.

Rules:
1. Answer ONLY from the CONTEXT passages provided. Never use outside knowledge.
2. If the context does not contain the answer, reply with exactly: {NOT_FOUND_TOKEN}
3. Cite the article number(s) you used, e.g. "(Article 39)".
4. Be clear and concise. Quote exact constitutional wording when precision matters,
   then explain it in plain English.
5. Use the conversation history only to understand what the user refers to."""


def result_label(r):
    return f"Article {r['article']}" + (f" — {r['heading']}" if r.get("heading") else "")


def build_context(results):
    return "\n\n".join(
        f"[{i}] {result_label(r)}\n{r['text']}"
        for i, r in enumerate(results, start=1)
    )


def call_gemini(prompt, system=None, max_tokens=8192):
    # Gemini 3.x: use thinking_level ("low" = fastest). Google says not to send
    # temperature/top_p/top_k, and the old thinking_budget is rejected.
    # Thinking tokens can use part of max_output_tokens, so keep it generous.
    config = types.GenerateContentConfig(
        system_instruction=system,
        max_output_tokens=max_tokens,
        thinking_config=types.ThinkingConfig(thinking_level="low"),
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )
    response = llm_client.models.generate_content(
        model=GEMINI_MODEL, contents=prompt, config=config
    )
    return (response.text or "").strip()


def extractive_answer(results, max_chars=900):
    best = results[0]
    text = best["text"]
    if len(text) > max_chars:
        text = text[:text.rfind(" ", 0, max_chars)] + " …"
    return f"According to {result_label(best)}:\n\n{text}"


def generate_answer(question, results, history_text=""):
    if llm_client is None:
        return extractive_answer(results)

    prompt = ""
    if history_text:
        prompt += f"CONVERSATION HISTORY:\n{history_text}\n\n"
    prompt += f"CONTEXT:\n{build_context(results)}\n\nQUESTION: {question}\n\nANSWER:"

    try:
        return call_gemini(prompt, system=SYSTEM_PROMPT)
    except Exception as e:
        print("Gemini error:", e)
        return extractive_answer(results)

In [ ]:
question = "What are the qualifications to become a member of Parliament?"

results = legal_retrieve(question, top_k=5)
print("Retrieved:", [result_label(r) for r in results])
print()
print(generate_answer(question, results))

# 9. Out-of-Scope Handling


The bot must not invent answers for questions outside the knowledge base. Three layers:

1. **Small talk** (hi / thanks / who are you) gets a friendly canned reply.
2. **Retrieval confidence:** if the best *semantic* score is low **and** few of the question's words appear in the retrieved
   articles (*coverage*), reply "not found". Borderline cases get a **clarification request** with suggested topics.
3. **LLM guard:** the prompt tells Gemini to answer `NOT_FOUND` when the context doesn't contain the answer.

Run the calibration cell first to see typical scores, then adjust the two thresholds if needed.

In [ ]:
# ============================================================
# CELL 12 — OUT-OF-SCOPE HANDLING
# ============================================================

NOT_FOUND_THRESHOLD = 0.22   # below this -> not found
CLARIFY_THRESHOLD = 0.32     # below this -> ask to clarify

NOT_FOUND_MESSAGE = (
    "Sorry, I couldn't find information about that in my knowledge base. "
    "I can only answer questions about the Constitution of Bangladesh. "
    "Try rephrasing, or ask about a topic such as fundamental rights, "
    "Parliament, the President, the judiciary or elections."
)

SMALLTALK = [
    (r"^(hi|hello|hey|salam|assalamu ?alaikum|good (morning|afternoon|evening))\b[\s!.]*$",
     "Hello! I'm BDCBot. Ask me anything about the Constitution of Bangladesh."),
    (r"^(thanks|thank you|thx|ok(ay)?|great|nice)\b[\s!.]*$",
     "You're welcome! Ask another question whenever you like."),
    (r"^((who|what) are you|what('?s| is) your name|what can you do|help)\b[\s?!.]*$",
     "I'm BDCBot. I answer questions using only the Constitution of Bangladesh and cite the article I used."),
]


def smalltalk_reply(question):
    q = question.strip().lower()
    for pattern, reply in SMALLTALK:
        if re.search(pattern, q):
            return reply
    return None


def query_coverage(query, results):
    """Share of the question's content words found in the retrieved articles."""
    terms = [t for t in (normalize_term(w) for w in get_query_terms(query)) if len(t) > 2]
    if not terms:
        return 1.0
    words = set()
    for r in results:
        words.update(normalize_text(f"{r.get('heading') or ''} {r['text']}").split())
    prefixes = {w[:5] for w in words}
    hits = sum(1 for t in terms if t in words or t[:5] in prefixes)
    return hits / len(terms)


def answer_question(question, history_text="", retrieval_query=None):
    reply = smalltalk_reply(question)
    if reply:
        return {"status": "smalltalk", "answer": reply, "sources": []}

    query = retrieval_query or question
    results = legal_retrieve(query, top_k=5)

    best_semantic = max((r["semantic_score"] for r in results), default=0.0)
    forced = any(r.get("legal_phrase_match") for r in results)
    coverage = query_coverage(query, results)

    if not results or (not forced and (best_semantic < NOT_FOUND_THRESHOLD or coverage < 0.25)):
        return {"status": "not_found", "answer": NOT_FOUND_MESSAGE, "sources": []}

    if not forced and (best_semantic < CLARIFY_THRESHOLD or coverage < 0.5):
        suggestions = "\n".join(f"- {result_label(r)}" for r in results[:3])
        return {"status": "clarify",
                "answer": "I'm not sure I found the right article. Did you mean:\n"
                          + suggestions + "\nPlease rephrase or add more detail.",
                "sources": [result_label(r) for r in results[:3]]}

    answer = generate_answer(query, results, history_text)
    if answer.strip().upper().startswith(NOT_FOUND_TOKEN):
        return {"status": "not_found", "answer": NOT_FOUND_MESSAGE, "sources": []}

    return {"status": "answered", "answer": answer,
            "sources": [result_label(r) for r in results]}

In [ ]:
# ------------------------------------------------------------
# Calibration: compare scores for in-scope vs out-of-scope questions
# ------------------------------------------------------------

in_scope = [
    "What is the state language of Bangladesh?",
    "What are the fundamental rights of citizens?",
    "Who appoints the Chief Justice?",
    "How is the President elected?",
]
out_of_scope = [
    "Who won the 2022 football World Cup?",
    "How do I bake a chocolate cake?",
    "What is the capital of France?",
    "Explain quantum computing",
]

for label, questions in [("IN-SCOPE", in_scope), ("OUT-OF-SCOPE", out_of_scope)]:
    print("\n" + label)
    for q in questions:
        res = legal_retrieve(q, top_k=5)
        best = max(r["semantic_score"] for r in res)
        print(f"  semantic={best:.3f}  coverage={query_coverage(q, res):.2f}  {q}")

In [ ]:
for q in ["Hello!", "What is the state religion?", "Who won the 2022 football World Cup?",
          "How do I bake a chocolate cake?"]:
    out = answer_question(q)
    print(f"\nQ: {q}\n[{out['status']}] {out['answer'][:400]}")

# 10. Conversation Memory

Short-term memory keeps the last few turns of the current session. Follow-up questions such as
*"Are there any restrictions on it?"* are rewritten into a standalone question (by Gemini, or by a simple
heuristic without a key) **before retrieval**, and the history is passed to the LLM for context.

In [ ]:
# ============================================================
# CELL 13 — CONVERSATION MEMORY (short-term, per session)
# ============================================================

from collections import deque

FOLLOW_UP_HINTS = re.compile(
    r"\b(it|its|this|that|these|those|they|them|their|he|she|his|her|"
    r"same|above|previous|also|more|else|another|again)\b"
    r"|^(and|what about|how about|why|explain|elaborate|tell me more|more)\b",
    re.I,
)

REWRITE_PROMPT = """Given the conversation and a follow-up question, rewrite the follow-up
as a standalone question. Keep it short. If it is already standalone, return it unchanged.
Return ONLY the question.

Conversation:
{history}

Follow-up question: {question}
Standalone question:"""


class ChatSession:

    def __init__(self, max_turns=6):
        self.history = deque(maxlen=max_turns)   # (role, text, sources)

    def history_text(self):
        return "\n".join(
            f"{'User' if role == 'user' else 'Assistant'}: {text[:600]}"
            for role, text, _ in self.history
        )

    def standalone_question(self, question):
        if not self.history:
            return question
        if len(question.split()) > 4 and not FOLLOW_UP_HINTS.search(question):
            return question

        if llm_client is not None:
            try:
                rewritten = call_gemini(
                    REWRITE_PROMPT.format(history=self.history_text(), question=question),
                    max_tokens=2048,
                )
                if rewritten:
                    return rewritten.splitlines()[0].strip()
            except Exception as e:
                print("Rewrite failed:", e)

        # Heuristic fallback: attach the previous topic
        last_sources = next((s for role, _, s in reversed(self.history) if role == "assistant" and s), None)
        last_user = next((t for role, t, _ in reversed(self.history) if role == "user"), "")
        topic = last_sources[0] if last_sources else last_user
        return f"{question} (regarding: {topic})"

    def ask(self, question):
        standalone = self.standalone_question(question)
        result = answer_question(question, self.history_text(), retrieval_query=standalone)
        self.history.append(("user", question, None))
        self.history.append(("assistant", result["answer"], result["sources"]))
        result["standalone_question"] = standalone
        return result

In [ ]:
session = ChatSession()

for q in [
    "What does Article 39 say?",
    "Are there any restrictions on it?",
    "What are the qualifications to become a member of Parliament?",
    "And what disqualifies someone?",
]:
    out = session.ask(q)
    print("\n" + "=" * 80)
    print("USER:      ", q)
    print("STANDALONE:", out["standalone_question"])
    print(f"BOT [{out['status']}]:")
    print(out["answer"][:700])

# 11. Export for the Local App

The local app (FastAPI backend + web frontend in this repository) uses the **same parser and retrieval code** as this
notebook, in `backend/app/`. You can either put the PDF in `backend/data/seed/`, or download the JSON below and upload it
from the app's **Knowledge base** page. Both give the same articles.

The exported JSON includes each article's `amendment_notes`, so uploading it gives the app the amendment history too.

In [ ]:
with open("constitution_hierarchy.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("Records:", len(records), "| Chunks:", len(chunks))

try:
    from google.colab import files
    files.download("constitution_hierarchy.json")
except ImportError:
    pass